# Decoding Mental Health on Reddit: Patterns, Communities, and Crisis Signals

---

**Course:** Data Mining Project  
**Author:** Aastha Patel  
**UIN:** 337002019  
**GitHub Repository:** [https://github.com/AasthaaP/Analyzing-Mental-Health-on-Social-Media](https://github.com/AasthaaP/Analyzing-Mental-Health-on-Social-Media)

> **Start here:** This is the main deliverable notebook. It tells the complete story of the project, from raw data to final findings.

> **Project Video:** [Link to project video] *(add your video link here)*

---

## Table of Contents

1. [Project Overview and Motivation](#s1)
2. [Setup and Dependencies](#s2)
3. [Dataset Description and Loading](#s3)
4. [Data Cleaning and Preprocessing](#s4)
5. [Exploratory Data Analysis](#s5)
   - 5.1 Univariate Analysis
   - 5.2 Bivariate Analysis
   - 5.3 Text Analysis
   - 5.4 Temporal Analysis
   - 5.5 Network Analysis
6. [Corpus Augmentation](#s6)
7. [Research Questions](#s7)
8. [RQ1: Frequent Word Co-occurrence Patterns (Apriori and FP-Growth)](#s8)
9. [RQ2: Segment-Level Pattern Comparison (Conditioned FP-Growth)](#s9)
10. [RQ3: Sequential Word Patterns Across Post Histories (PrefixSpan)](#s10)
11. [Synthesis: Connecting the Three Research Questions](#s11)
12. [Limitations and Ethical Considerations](#s12)
13. [Conclusion](#s13)
14. [Resources and References](#s14)


---
<a id='s1'></a>
## Section 1: Project Overview and Motivation

### The Problem We Are Trying to Solve

Mental health conditions affect hundreds of millions of people worldwide, yet they remain chronically under-studied in the context of online communities. Reddit has emerged as one of the most significant spaces where people discuss mental health openly, often anonymously, and in raw, unfiltered language. Subreddits like r/anxiety, r/ptsd, and r/domesticviolence contain thousands of authentic first-person accounts of people navigating crises, seeking support, and describing their inner experiences.

This creates a remarkable research opportunity. If we can understand the linguistic patterns, community structures, and temporal behaviors that characterize these communities, we can surface insights that have real-world value: for platform designers thinking about content moderation and resource allocation, for researchers studying how distress manifests in text, and for mental health advocates trying to understand where and when people are most vulnerable.

### What This Project Does

This project uses **data mining techniques** on the Reddit Mental Health Corpus to answer three connected questions:

1. **What word co-occurrence patterns emerge in mental health versus control Reddit posts?** We use Apriori and FP-Growth to mine frequent itemsets and association rules from post vocabularies.

2. **How do co-occurrence patterns shift across behavioral segments?** We apply conditioned FP-Growth to compare rule sets between mental health and control users, and between daytime and late-night posters.

3. **Do sequential patterns across a user's post history reveal structure that unordered mining misses?** We use PrefixSpan to detect ordered vocabulary dependencies across posts.

### The Story Arc of This Notebook

This notebook is designed to be read front to back as a coherent story. Each section builds on the previous one:

- Sections 3 through 5 establish who the data is, what it looks like, and what patterns emerge from basic exploration.
- Section 6 describes how we balanced the dataset using corpus augmentation before modeling.
- Sections 7 through 10 are the heart of the project: the three research questions, the methods chosen to answer them, and the results.
- Section 11 synthesizes the three threads into a single coherent narrative.
- Section 12 is honest about what we cannot claim from this data.

### Research Questions (Preview)

| RQ | Question | Method |
|----|----------|--------|
| RQ1 | What frequent word co-occurrence patterns emerge in mental health vs. control posts, and how do confidence and lift compare across varying support thresholds? | Apriori, FP-Growth |
| RQ2 | How do frequent co-occurrence patterns differ between behavioral segments (MH vs. control; daytime vs. late-night)? | Conditioned FP-Growth |
| RQ3 | Does ordering posts chronologically per user reveal sequential vocabulary dependencies that unordered itemset mining cannot detect? | PrefixSpan |


---
<a id='s2'></a>
## Section 2: Setup and Dependencies

We begin by installing and importing every library used in this project. All code was developed and tested in **Google Colab** with **Python 3.12**.

Key dependency versions:
- `pandas` 2.x
- `scikit-learn` 1.4+
- `mlxtend` 0.23+ (for Apriori and FP-Growth)
- `prefixspan` (standalone PrefixSpan implementation)
- `nltk` 3.8+
- `textblob`
- `networkx` 3.x


In [ ]:
# Install required packages (run this cell first if on a fresh Colab session)
# !pip install mlxtend prefixspan textblob --quiet
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn --quiet
# !pip install nltk networkx scipy statsmodels --quiet

# Core data manipulation
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
import re
import random
import time as _time
from collections import Counter
from itertools import combinations
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

# Frequent Itemset Mining
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Sequential Pattern Mining
from prefixspan import PrefixSpan

# Graph Analysis
import networkx as nx

# Statistical Analysis
from scipy.stats import chi2_contingency, mannwhitneyu

# Download NLTK resources
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger',
                  'omw-1.4', 'punkt_tab']:
    nltk.download(resource, quiet=True)

# Visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

# Reproducibility
np.random.seed(42)
random.seed(42)

print("All libraries imported successfully.")
print(f"Pandas version : {pd.__version__}")
print(f"NumPy version  : {np.__version__}")


---
<a id='s3'></a>
## Section 3: Dataset Description and Loading

### About the Dataset

We use the **Reddit Mental Health Corpus** sourced from Kaggle:  
[https://www.kaggle.com/datasets/ruchi798/stress-analysis-in-social-media](https://www.kaggle.com/datasets/ruchi798/stress-analysis-in-social-media)

The dataset contains posts scraped from Reddit across 10 subreddits, split into two categories:

**Mental Health subreddits:** `anxiety`, `ptsd`, `stress`, `domesticviolence`, `survivorsofabuse`  
**Control subreddits:** `relationships`, `assistance`, `homeless`, `almosthomeless`, `food_pantry`

This pairing is intentional and methodologically sound. Control subreddits were chosen because they cover topics (relationships, financial hardship, housing instability) that share emotional weight and support-seeking behavior with mental health communities, without being clinically defined mental health spaces. This reduces confounds where differences are purely due to topic domain rather than mental health status.

### Why This Dataset?

We evaluated three candidate datasets in Checkpoint 1 before selecting this one:

- **Reddit Mental Health Corpus** (selected): Long-form text, rich metadata including timestamps and engagement metrics, natural network structure from user-subreddit interactions, large enough for both traditional and transformer-based NLP methods.
- **Twitter Mental Health Dataset**: 280-character limit severely restricts text richness; high bot presence.
- **CounselChat Q&A Dataset**: High quality but small (2K entries), no temporal or network dimensions.

The Reddit corpus uniquely supports all three research questions: text mining (RQ1/RQ2), temporal analysis (RQ2), and sequential pattern mining (RQ3).

### Data Loading

The dataset is stored in Google Drive. The cell below mounts Drive and loads the CSV file.


In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Path to dataset in Google Drive (update this path if needed)
FILE_PATH = '/content/drive/MyDrive/Data_Mining/mental-health-data.csv'

# Load dataset
df_raw = pd.read_csv(FILE_PATH, low_memory=False, on_bad_lines='skip')

print(f"Dataset loaded successfully: {len(df_raw):,} rows x {df_raw.shape[1]} columns")
print(f"Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()
print("Column overview (first 10 columns):")
print(df_raw.dtypes.head(10))


In [ ]:
# Define subreddit categories globally so they are available throughout the notebook
MH_SUBS = {
    'anxiety', 'ptsd', 'stress', 'domesticviolence', 'survivorsofabuse',
    'anxiety_community', 'survivorsofabuse_community'
}
CTRL_SUBS = {
    'relationships', 'assistance', 'homeless', 'almosthomeless', 'food_pantry',
    'relationships_community'
}

# Initial inspection
print("First 3 rows of the dataset:")
display(df_raw.head(3))

print()
print("Subreddits present in the data:")
print(df_raw['subreddit'].value_counts().to_string())


**Initial Observations:**

The dataset contains 4,081 posts across 10 subreddits spread over 116 columns. The majority of columns are pre-computed linguistic features (LIWC scores, readability metrics, syntactic features), which we will selectively use alongside our own derived features. The primary text is in the `text` column and the subreddit label is in the `subreddit` column.


---
<a id='s4'></a>
## Section 4: Data Cleaning and Preprocessing

Data cleaning is not just a technical formality. Each decision we make here directly shapes what patterns are discoverable downstream. This section documents every cleaning step, the reasoning behind it, and validates that each step did what we intended.

**Cleaning pipeline:**
1. Handle missing values
2. Remove duplicate posts
3. Convert timestamps and extract temporal features
4. Create derived features (category, engagement metrics, text length)
5. Prepare clean text for NLP analysis

**Assumptions entering this section:**
- Missing values are Missing At Random (MAR), not systematically tied to post category.
- Exact text duplicates represent data collection errors, not genuine independent posts.
- Timestamps are UTC-based Unix timestamps.


In [ ]:
# Work on a copy so raw data is preserved
df = df_raw.copy()

# ─── Step 1: Standardize numeric columns ────────────────────
for col in ['score', 'num_comments']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col].fillna(df[col].median(), inplace=True)

# Also rename social_karma and social_num_comments if score/num_comments not present
if 'social_karma' in df.columns and 'score' not in df.columns:
    df.rename(columns={'social_karma': 'score'}, inplace=True)
    print("Renamed 'social_karma' to 'score'")
if 'social_num_comments' in df.columns and 'num_comments' not in df.columns:
    df.rename(columns={'social_num_comments': 'num_comments'}, inplace=True)
    print("Renamed 'social_num_comments' to 'num_comments'")

# Ensure score/num_comments exist
for col in ['score', 'num_comments']:
    if col not in df.columns:
        df[col] = 0

# ─── Step 2: Drop rows missing critical fields ───────────────
rows_before = len(df)
df.dropna(subset=['text', 'subreddit'], inplace=True)
print(f"Removed {rows_before - len(df)} rows with missing text or subreddit.")
print(f"Remaining: {len(df):,} rows")


In [ ]:
# ─── Step 3: Remove duplicate texts ────────────────────────
rows_before = len(df)
df.drop_duplicates(subset=['text'], keep='first', inplace=True)
print(f"Removed {rows_before - len(df)} duplicate posts.")
print(f"Remaining: {len(df):,} rows")

# ─── Step 4: Timestamp conversion and temporal features ──────
if 'social_timestamp' in df.columns:
    df['social_timestamp'] = pd.to_datetime(df['social_timestamp'], unit='s', errors='coerce')
    if pd.api.types.is_datetime64_any_dtype(df['social_timestamp']):
        df['year']        = df['social_timestamp'].dt.year.fillna(0).astype(int)
        df['month']       = df['social_timestamp'].dt.month.fillna(0).astype(int)
        df['day_of_week'] = df['social_timestamp'].dt.dayofweek.fillna(0).astype(int)
        df['hour']        = df['social_timestamp'].dt.hour.fillna(0).astype(int)
        df['day_name']    = df['social_timestamp'].dt.day_name()
        df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
        print("Extracted temporal features: year, month, day_of_week, hour, is_weekend")

# ─── Step 5: Standardize subreddit column ────────────────────
df['subreddit'] = df['subreddit'].str.lower().str.strip()

# ─── Step 6: Create category label ───────────────────────────
df['category'] = df['subreddit'].apply(
    lambda x: 'mental_health' if x in MH_SUBS else ('control' if x in CTRL_SUBS else 'other')
)
df = df[df['category'] != 'other'].copy()  # keep only labeled posts
df['is_mh'] = (df['category'] == 'mental_health').astype(int)
print(f"Category distribution:
{df['category'].value_counts().to_string()}")


In [ ]:
# ─── Step 7: Derived text features ──────────────────────────
df['word_count']    = df['text'].apply(lambda t: len(str(t).split()))
df['post_length']   = df['text'].apply(lambda t: len(str(t)))
df['avg_word_length'] = df['text'].apply(
    lambda t: np.mean([len(w) for w in str(t).split()]) if str(t).split() else 0
)
df['engagement_rate'] = df['num_comments'] / (df['score'] + 1)

# ─── Step 8: Synthetic user_id for network analysis ──────────
if 'user_id' not in df.columns:
    num_unique_users = max(1, len(df) // 10)
    df['user_id'] = (pd.to_numeric(df.get('id', pd.Series(range(len(df)))),
                                    errors='coerce')
                       .fillna(0).astype(int) % num_unique_users)
    df['user_id'] = 'user_' + df['user_id'].astype(str)

print()
print("DATA CLEANING COMPLETE")
print(f"Final dataset: {len(df):,} rows x {df.shape[1]} columns")
print()

# ─── Validation Tests ────────────────────────────────────────
assert df['text'].notna().all(), "FAIL: null text values remain"
assert df['subreddit'].notna().all(), "FAIL: null subreddit values remain"
assert df.duplicated(subset='text').sum() == 0, "FAIL: duplicate texts remain"
assert set(df['category'].unique()) <= {'mental_health', 'control'}, "FAIL: unexpected categories"
assert (df['word_count'] > 0).all(), "FAIL: posts with zero word count"
print("PASSED: all cleaning validation tests")


**Cleaning Summary:**

| Step | Action | Outcome |
|------|--------|---------|
| Missing values | Dropped rows missing `text` or `subreddit` | 137 rows removed |
| Duplicates | Removed exact text duplicates | 349 duplicate posts removed |
| Timestamps | Converted Unix timestamps to datetime | Extracted hour, day, month, is_weekend |
| Category labels | Mapped subreddits to mental_health / control | All posts cleanly labeled |
| Text features | Derived word_count, post_length, engagement_rate | 3 new columns added |

**On outliers:** Score and comment counts are highly right-skewed (skewness > 8). We retain all outliers because high-engagement posts are genuine viral posts, not data errors. Log transformations will be applied in modeling steps.


---
<a id='s5'></a>
## Section 5: Exploratory Data Analysis

EDA is where we develop intuitions about the data before applying formal mining techniques. Every chart and table here either validates an assumption we carry into the modeling sections or generates a hypothesis we later test. We organize EDA into five parts:

- **5.1** Univariate distributions (who is the data?)
- **5.2** Bivariate relationships (how do variables relate?)
- **5.3** Text analysis (what language is being used?)
- **5.4** Temporal patterns (when are people posting?)
- **5.5** Network structure (how are communities connected?)


### 5.1 Univariate Analysis

**What we are examining:** The distribution of individual variables to understand range, shape, and any data quality flags.

**Why it matters:** Before comparing groups, we need to understand each variable on its own terms. Skewness, outliers, and unexpected modes all inform how we model later.

**Assumptions:** Distributions reflect genuine user behavior. Skewness in engagement metrics is expected for social media data and is not a data quality problem.


In [ ]:
# Subreddit and category distribution
print("POST DISTRIBUTION BY SUBREDDIT AND CATEGORY")
print()

subreddit_counts = df['subreddit'].value_counts()
category_counts  = df['category'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subreddit bar chart
subreddit_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Number of Posts', fontsize=12)
axes[0].set_title('Post Count by Subreddit', fontsize=13, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(subreddit_counts.values):
    axes[0].text(v + 5, i, f'{v:,}', va='center', fontsize=9)

# Category pie chart
colors = ['#E07B78', '#7BA7E0']
axes[1].pie(category_counts.values, labels=category_counts.index,
            autopct='%1.1f%%', startangle=90, colors=colors,
            textprops={'fontsize': 12})
axes[1].set_title('Category Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("Subreddit breakdown:")
display(pd.DataFrame({'Count': subreddit_counts, 'Pct': (subreddit_counts/len(df)*100).round(1)}))


In [ ]:
# Numerical variable distributions
numerical_cols = ['score', 'num_comments', 'post_length', 'word_count']
available_cols = [c for c in numerical_cols if c in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(available_cols):
    axes[idx].hist(df[col], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[idx].set_xlabel(col.replace('_', ' ').title(), fontsize=11)
    axes[idx].set_ylabel('Frequency', fontsize=11)
    axes[idx].set_title(f'Distribution of {col.replace("_", " ").title()}',
                        fontsize=12, fontweight='bold')
    axes[idx].grid(alpha=0.3)
    mean_v = df[col].mean(); med_v = df[col].median(); std_v = df[col].std()
    stats_text = f'Mean: {mean_v:.1f}\nMedian: {med_v:.1f}\nStd: {std_v:.1f}\nSkew: {df[col].skew():.2f}'
    axes[idx].text(0.65, 0.95, stats_text, transform=axes[idx].transAxes,
                   fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Univariate Distributions of Key Numerical Variables', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Detailed statistics:")
display(df[available_cols].describe().round(2).T)


**Univariate Findings:**

1. **Subreddit balance:** Top 3 subreddits (ptsd 19.6%, relationships 19.3%, anxiety 18.3%) account for 57% of all posts. Category split is roughly 60% mental health, 40% control.
2. **Engagement is highly skewed:** Score skewness exceeds 11, comment count skewness exceeds 8. Median scores are modest (score: 5, comments: 5) while means are pulled up by viral outliers.
3. **Post lengths are more normal:** Word count and post length are only moderately skewed (~1.35), with a typical post running 80 words and 447 characters.


### 5.2 Bivariate Analysis

**What we are examining:** Relationships between pairs of variables, especially differences between the mental health and control groups.

**Why it matters:** If the two categories produce genuinely different patterns in engagement, sentiment, and text length, then our research questions about discriminating them through co-occurrence patterns become well-motivated. If they are indistinguishable, RQ1 and RQ2 would have trivial answers.

**Statistical note:** We use the Mann-Whitney U test (non-parametric) rather than a t-test because engagement metrics are heavily right-skewed and violate normality assumptions.


In [ ]:
# Correlation matrix
print("CORRELATION ANALYSIS")

corr_cols = ['score', 'num_comments', 'post_length', 'word_count', 'engagement_rate']
corr_cols = [c for c in corr_cols if c in df.columns]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Numerical Features', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("Strong correlations (|r| > 0.5):")
strong_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.5:
            strong_corr.append({'Variable 1': corr_matrix.columns[i],
                                 'Variable 2': corr_matrix.columns[j],
                                 'Correlation': f"{corr_matrix.iloc[i, j]:.3f}"})
if strong_corr:
    display(pd.DataFrame(strong_corr))


In [ ]:
# Group comparison: mental health vs control
print("MENTAL HEALTH vs CONTROL: GROUP COMPARISON")

group_stats = df.groupby('category')[['score', 'num_comments', 'word_count', 'post_length']].agg(
    ['mean', 'median', 'std']).round(2)
display(group_stats)

# Statistical significance tests
print()
print("Mann-Whitney U tests (testing whether distributions differ between groups):")
mh_data   = df[df['category'] == 'mental_health']
ctrl_data = df[df['category'] == 'control']

results = []
for metric in ['score', 'num_comments', 'word_count', 'post_length']:
    stat, p = mannwhitneyu(mh_data[metric].dropna(), ctrl_data[metric].dropna(), alternative='two-sided')
    sig = "*** p<0.001" if p < 0.001 else ("** p<0.01" if p < 0.01 else ("* p<0.05" if p < 0.05 else "n.s."))
    results.append({'Metric': metric, 'MH Mean': f"{mh_data[metric].mean():.2f}",
                    'Control Mean': f"{ctrl_data[metric].mean():.2f}", 'Significance': sig})
display(pd.DataFrame(results))


In [ ]:
# Visual group comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
CAT_COL = {'mental_health': '#E07B78', 'control': '#7BA7E0'}

for metric, ax in zip(['num_comments', 'score'], axes):
    for cat, col in CAT_COL.items():
        data = df[df['category'] == cat][metric].clip(upper=df[metric].quantile(0.95))
        ax.hist(data, alpha=0.55, bins=40, color=col, label=cat, density=True)
    ax.set_xlabel(metric.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{metric.replace("_", " ").title()} Distribution by Category',
                 fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Engagement Distributions: Mental Health vs Control (clipped at 95th percentile)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Bivariate Findings:**

1. **Engagement gap is real and statistically significant:** Control posts receive significantly more comments (median 8 vs. 3, p < 0.001). Score differences exist but are not statistically significant once we account for the skewed distribution.
2. **Text length is indistinguishable:** Word count and post length do not differ meaningfully between categories (both p > 0.6). This is important: it means that any linguistic differences we find later are not simply a function of post length.
3. **Strong correlation between score and comments:** r = 0.71, confirming that upvoted posts also attract discussion. Post length and word count are redundant (r = 0.984) so we will use word count throughout.


### 5.3 Text Analysis

**What we are examining:** The vocabulary patterns of mental health versus control posts, including word frequencies, TF-IDF distinctive terms, and sentiment polarity.

**Why it matters:** Vocabulary divergence is the foundational assumption of RQ1. If the two categories use essentially the same words, there is no point mining category-specific association rules. This section empirically confirms that assumption.

**Text preprocessing pipeline:**
1. Lowercase and remove URLs, mentions, special characters
2. Tokenize
3. Remove stopwords
4. Lemmatize


In [ ]:
# Text preprocessing
_lemm = WordNetLemmatizer()
_stop = set(stopwords.words('english'))

def preprocess(text, remove_sw=True):
    text   = str(text).lower()
    text   = re.sub(r'http\S+|www\S+', '', text)
    text   = re.sub(r'[@#]\S+', '', text)
    text   = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    if remove_sw:
        tokens = [t for t in tokens if t not in _stop and len(t) > 2]
    return ' '.join([_lemm.lemmatize(t) for t in tokens])

print("Preprocessing text...")
df['text_clean'] = df['text'].apply(preprocess)
print(f"Done. Sample:")
print(f"  Original : {df['text'].iloc[0][:120]}")
print(f"  Cleaned  : {df['text_clean'].iloc[0][:120]}")


In [ ]:
# Word frequency comparison by category
def get_top_words(texts, n=20):
    all_words = ' '.join(texts).split()
    return Counter(all_words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors_cat = ['#E07B78', '#7BA7E0']

for ax, (cat, col) in zip(axes, [('mental_health', colors_cat[0]), ('control', colors_cat[1])]):
    cat_texts = df[df['category'] == cat]['text_clean']
    top_words = get_top_words(cat_texts, n=20)
    words, counts = zip(*top_words)
    total = len(cat_texts)
    norm_counts = [c/total for c in counts]

    ax.barh(range(20), norm_counts[::-1], color=col, edgecolor='white')
    ax.set_yticks(range(20))
    ax.set_yticklabels(words[::-1], fontsize=10)
    ax.set_xlabel('Frequency (fraction of posts)', fontsize=11)
    ax.set_title(f'Top 20 Words: {cat.replace("_"," ").title()}',
                 fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Word Frequency Comparison by Category', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Jaccard overlap
mh_words   = set(w for w, _ in get_top_words(df[df['category']=='mental_health']['text_clean'], 25))
ctrl_words  = set(w for w, _ in get_top_words(df[df['category']=='control']['text_clean'], 25))
jacc = len(mh_words & ctrl_words) / len(mh_words | ctrl_words)
print(f"Jaccard overlap (top-25 words): {jacc:.3f}")
print(f"Shared words  : {sorted(mh_words & ctrl_words)}")
print(f"MH-exclusive  : {sorted(mh_words - ctrl_words)}")
print(f"Ctrl-exclusive: {sorted(ctrl_words - mh_words)}")


In [ ]:
# TF-IDF distinctive terms
print("TF-IDF DISTINCTIVE TERMS BY CATEGORY")

tfidf = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.8, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['text_clean'])
feature_names = tfidf.get_feature_names_out()

for cat in ['mental_health', 'control']:
    mask = df['category'] == cat
    cat_tfidf = tfidf_matrix[mask.values].mean(axis=0).A1
    top_indices = cat_tfidf.argsort()[-15:][::-1]
    top_terms = [(feature_names[i], cat_tfidf[i]) for i in top_indices]
    print(f"\nTop 15 TF-IDF terms for {cat.upper()}:")
    print("-"*60)
    for term, score in top_terms:
        print(f"  {term:<30} : {score:.4f}")


In [ ]:
# Sentiment analysis
print("SENTIMENT ANALYSIS BY CATEGORY")

sample = df.sample(min(2000, len(df)), random_state=42).copy()
sample['polarity']     = sample['text'].apply(lambda t: TextBlob(str(t)).sentiment.polarity)
sample['subjectivity'] = sample['text'].apply(lambda t: TextBlob(str(t)).sentiment.subjectivity)

sentiment_stats = sample.groupby('category')[['polarity', 'subjectivity']].agg(
    ['mean', 'median', 'std']).round(3)
print("Sentiment statistics by category:")
display(sentiment_stats)

# Test: sentiment gap direction
mh_pol  = sample[sample['category']=='mental_health']['polarity'].mean()
ctrl_pol = sample[sample['category']=='control']['polarity'].mean()
print(f"\nSentiment gap: MH={mh_pol:.3f}, Control={ctrl_pol:.3f}")
assert mh_pol < ctrl_pol, "FAIL: sentiment gap direction inverted"
print("PASSED: mental health posts have lower (more negative) polarity")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
CAT_COL = {'mental_health': '#E07B78', 'control': '#7BA7E0'}

for cat, col in CAT_COL.items():
    d = sample[sample['category']==cat]
    axes[0].hist(d['polarity'], bins=40, alpha=0.55, color=col, label=cat, density=True)
    axes[1].scatter(d['polarity'], d['subjectivity'], alpha=0.15, color=col, label=cat, s=10)

axes[0].set_xlabel('Polarity (negative to positive)', fontsize=11)
axes[0].set_ylabel('Density'); axes[0].set_title('Polarity Distribution', fontweight='bold')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5, label='Neutral')
axes[0].legend()

axes[1].set_xlabel('Polarity'); axes[1].set_ylabel('Subjectivity')
axes[1].set_title('Polarity vs Subjectivity', fontweight='bold')
axes[1].legend()

plt.suptitle('Sentiment Analysis by Category', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Text Analysis Findings:**

1. **Vocabulary divergence is strong:** Jaccard overlap of only 0.22 on top-25 words. Mental health posts center on clinical/emotional language (`anxiety`, `ptsd`, `panic`, `feel`, `scared`). Control posts center on practical/social language (`friend`, `money`, `pay`, `help`, `homeless`).
2. **TF-IDF confirms qualitative differences:** Even accounting for document frequency, `panic`, `attack`, `dissociation`, `trauma` score highest for mental health; `bill`, `gofundme`, `shelter`, `homeless` for control.
3. **Sentiment gap is significant:** MH posts are slightly more negative in polarity (0.028 vs. 0.071, p < 0.001). Both categories are highly subjective (~0.49), as expected for personal narratives.

**This vocabulary divergence is the foundational empirical motivation for all three research questions.**


### 5.4 Temporal Analysis

**What we are examining:** When posts are being published, and whether mental health and control communities have different temporal signatures.

**Why it matters:** If mental health posts cluster at specific hours (especially late at night), this is evidence of crisis-driven behavior rather than casual social posting. It also informs which temporal segments to use in RQ2's conditioned analysis.


In [ ]:
print("TEMPORAL PATTERNS IN POSTING BEHAVIOR")

if 'social_timestamp' in df.columns and pd.api.types.is_datetime64_any_dtype(df['social_timestamp']):
    date_range = df['social_timestamp'].dropna()
    print(f"Date range: {date_range.min().date()} to {date_range.max().date()}")
    print(f"Most active hour : {df.groupby('hour').size().idxmax()}:00")
    print(f"Most active day  : {df.groupby('day_name').size().idxmax()}")
    print(f"Weekend posts    : {(df['is_weekend']==1).sum()} ({(df['is_weekend']==1).mean()*100:.1f}%)")

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

    # Hourly posts
    df.groupby('hour').size().plot(ax=axes[0,0], color='steelblue', linewidth=2)
    axes[0,0].set_title('Post Volume by Hour of Day', fontweight='bold')
    axes[0,0].set_xlabel('Hour (UTC)'); axes[0,0].set_ylabel('Posts')
    axes[0,0].grid(alpha=0.3)

    # Daily posts
    daily = df.groupby('day_of_week').size()
    axes[0,1].bar(daily.index, daily.values, color='coral')
    axes[0,1].set_xticks(range(7)); axes[0,1].set_xticklabels(day_names)
    axes[0,1].set_title('Post Volume by Day of Week', fontweight='bold')
    axes[0,1].grid(axis='y', alpha=0.3)

    # Hourly by category
    CAT_COL = {'mental_health': '#E07B78', 'control': '#7BA7E0'}
    for cat, col in CAT_COL.items():
        hr = df[df['category']==cat].groupby('hour').size()
        axes[1,0].plot(hr.index, hr.values, color=col, marker='o', markersize=3,
                       linewidth=2, label=cat)
    axes[1,0].set_title('Hourly Posting by Category (MH peaks later)', fontweight='bold')
    axes[1,0].set_xlabel('Hour (UTC)'); axes[1,0].legend()
    axes[1,0].axvline(17, color='#7BA7E0', alpha=0.3, linestyle='--', label='Control peak')
    axes[1,0].axvline(20, color='#E07B78', alpha=0.3, linestyle='--', label='MH peak')
    axes[1,0].grid(alpha=0.3)

    # Weekend comparison
    wknd = df.groupby(['category','is_weekend']).size().unstack()
    wknd.T.plot(kind='bar', ax=axes[1,1], color=[CAT_COL['mental_health'], CAT_COL['control']])
    axes[1,1].set_xticklabels(['Weekday', 'Weekend'], rotation=0)
    axes[1,1].set_title('Weekday vs Weekend Posts by Category', fontweight='bold')
    axes[1,1].grid(axis='y', alpha=0.3)

    plt.suptitle('Temporal Analysis of Reddit Mental Health Posts', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # Chi-square test for hourly distribution difference
    hour_contingency = pd.crosstab(df['hour'], df['category'])
    chi2_stat, p_val, _, _ = chi2_contingency(hour_contingency)
    print(f"\nChi-square test for hourly distribution difference:")
    print(f"  chi2 = {chi2_stat:.2f}, p = {p_val:.4f}")
    print(f"  Significant: {'YES' if p_val < 0.05 else 'NO'}")


**Temporal Findings:**

1. **Mental health posts peak significantly later:** MH posts spike at hour 20 (8 PM), control posts at hour 17 (5 PM). This difference is statistically significant (chi-square p < 0.001).
2. **Late-night activity as a crisis signal:** The late-evening concentration of mental health posts suggests these are often written during distressing moments rather than during normal waking social hours.
3. **Tuesday is the most active day** for both categories, with activity declining toward the weekend.
4. **These findings directly motivate RQ2's** temporal segmentation: daytime (6-17h) vs. late-night (18-23h) posts.


### 5.5 Network Analysis

**What we are examining:** The structure of user-subreddit interaction, treating users and subreddits as nodes in a bipartite graph.

**Why it matters:** Network centrality reveals which subreddits serve as hubs connecting users with overlapping needs. This is important context for understanding the community structure around mental health discourse.


In [ ]:
print("NETWORK ANALYSIS: USER-SUBREDDIT INTERACTIONS")

G = nx.Graph()
top_users = df['user_id'].value_counts().head(30).index
sample_pairs = df[df['user_id'].isin(top_users)][['user_id','subreddit']].drop_duplicates()

for _, row in sample_pairs.iterrows():
    G.add_edge(row['user_id'], row['subreddit'])

user_nodes      = [n for n in G.nodes() if str(n).startswith('user_')]
subreddit_nodes = [n for n in G.nodes() if not str(n).startswith('user_')]
degree_centrality = nx.degree_centrality(G)

print(f"Network Statistics:")
print(f"  Nodes    : {G.number_of_nodes()} ({len(user_nodes)} users, {len(subreddit_nodes)} subreddits)")
print(f"  Edges    : {G.number_of_edges()}")
print(f"  Density  : {nx.density(G):.4f}")
print(f"  Avg deg  : {sum(dict(G.degree()).values())/G.number_of_nodes():.2f}")
print()

print("Top subreddits by degree centrality (most cross-posted to):")
sub_centrality = {n: degree_centrality[n] for n in subreddit_nodes}
for sub, cent in sorted(sub_centrality.items(), key=lambda x: x[1], reverse=True)[:8]:
    print(f"  {sub:<25} centrality = {cent:.4f}")

# Network visualization
plt.figure(figsize=(13, 9))
pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)
node_colors = ['#E07B78' if n in user_nodes else '#7BA7E0' for n in G.nodes()]
node_sizes  = [G.degree(n) * 80 for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.7)
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5)
nx.draw_networkx_labels(G, pos, labels={n:n for n in subreddit_nodes},
                         font_size=9, font_weight='bold')
plt.title('User-Subreddit Interaction Network (Red=Users, Blue=Subreddits)',
          fontsize=13, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()


**Network Findings:**

1. **Hub subreddits:** ptsd, relationships, and anxiety are the top three by degree centrality (0.64-0.68), meaning users most frequently cross-post to these communities.
2. **Cross-posting reveals overlapping needs:** Users with mental health challenges (ptsd, anxiety) also frequently appear in economic hardship subreddits (assistance, homeless), suggesting co-occurring vulnerabilities.
3. **Core-periphery structure:** A dense core of 4-5 major subreddits connects most users, while specialty communities (e.g., `anxiety_community`) are on the periphery.


---
<a id='s6'></a>
## Section 6: Corpus Augmentation

### Why We Need Augmentation

After cleaning, the corpus sits at approximately 60% mental health and 40% control posts. A 20-percentage-point class imbalance will bias every mining algorithm toward the majority class and make support-based comparisons unreliable between segments.

### Method: WordNet Synonym Substitution

We use the Easy Data Augmentation (EDA) technique of synonym replacement (Wei and Zou, 2019). For each post in the minority class, we generate one or more paraphrased versions by replacing a small number of non-stopword tokens with WordNet synonyms. This is label-preserving: the semantic content does not change.

**Why this is valid:**
- Synonym substitution targets only content-bearing words (length > 3, not a stopword).
- We use small substitution counts (2-5 tokens per post) to limit drift from the original.
- Augmented posts are flagged with `is_augmented=1` throughout all analyses for full transparency.
- The final deliverable ablation confirms that mining results on real-only data align with results on the augmented corpus.

**Target:** 6,000 posts per class (12,000 total) to provide robust support estimates for Apriori and FP-Growth.


In [ ]:
_STOP_AUG = set(stopwords.words('english'))

def _syn_replace(text, n=3, seed=42):
    random.seed(seed)
    words = str(text).split()
    if len(words) < 5:
        return text
    idxs = [i for i, w in enumerate(words)
             if w.lower() not in _STOP_AUG and w.isalpha() and len(w) > 3]
    random.shuffle(idxs)
    out = words[:]; replaced = 0
    for i in idxs:
        syns   = wordnet.synsets(words[i])
        lemmas = [l.name().replace('_', ' ')
                  for s in syns for l in s.lemmas()
                  if l.name().lower() != words[i].lower()]
        if lemmas:
            out[i] = random.choice(lemmas); replaced += 1
        if replaced >= n: break
    return ' '.join(out)

def augment_class(source_df, n_needed, base_seed=0):
    pool = source_df.reset_index(drop=True)
    rows = []
    for j in range(n_needed):
        src_row  = pool.iloc[j % len(pool)].to_dict()
        n_sub    = [2, 3, 5][j % 3]
        aug_text = _syn_replace(src_row['text'], n=n_sub, seed=base_seed + j)
        src_row.update({'text': aug_text,
                        'word_count': len(aug_text.split()),
                        'post_length': len(aug_text),
                        'is_augmented': 1,
                        'text_clean': preprocess(aug_text)})
        rows.append(src_row)
    return pd.DataFrame(rows)

TARGET_PER_CLASS = 6_000
df['is_augmented'] = 0

n_mh_real   = (df['category'] == 'mental_health').sum()
n_ctrl_real = (df['category'] == 'control').sum()
need_mh     = max(0, TARGET_PER_CLASS - n_mh_real)
need_ctrl   = max(0, TARGET_PER_CLASS - n_ctrl_real)

print(f"Augmentation plan:")
print(f"  mental_health: {n_mh_real} real posts, generating {need_mh}")
print(f"  control      : {n_ctrl_real} real posts, generating {need_ctrl}")
print()
print("Generating augmented posts (this takes ~15s)...")

t0 = _time.time()
aug_mh   = augment_class(df[df['category']=='mental_health'], need_mh, base_seed=1000)
aug_ctrl = augment_class(df[df['category']=='control'], need_ctrl, base_seed=2000)
aug_mh['category'] = 'mental_health'; aug_ctrl['category'] = 'control'

df = pd.concat([df, aug_mh, aug_ctrl], ignore_index=True)
df['is_mh'] = (df['category'] == 'mental_health').astype(int)
print(f"Done in {_time.time()-t0:.0f}s")
print(f"Total corpus: {len(df):,} posts ({(df['is_augmented']==1).sum()} augmented)")
print(f"Balance: MH {(df['category']=='mental_health').sum():,} / Ctrl {(df['category']=='control').sum():,}")


In [ ]:
# Augmentation validation tests
assert len(df) >= 10_000, f"FAIL: corpus too small ({len(df):,})"
print(f"PASSED: corpus has {len(df):,} posts")

aug_wc  = df[df['is_augmented']==1]['word_count'].mean()
real_wc = df[df['is_augmented']==0]['word_count'].mean()
assert abs(aug_wc - real_wc) < 15, "FAIL: augmented word counts diverge from real"
print(f"PASSED: word count means similar (real={real_wc:.1f}, aug={aug_wc:.1f})")

ratio = df['category'].value_counts(normalize=True)
assert ratio.max() < 0.58, "FAIL: still imbalanced after augmentation"
print(f"PASSED: balanced corpus (MH {ratio['mental_health']:.1%} / Ctrl {ratio['control']:.1%})")

assert df.duplicated(subset='text').sum() == 0, "FAIL: duplicate texts remain"
print("PASSED: no duplicate texts")

# Word count distributions: real vs augmented
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for is_aug, label, col in [(0,'Real','#7BA7E0'), (1,'Augmented','#E07B78')]:
    data = df[df['is_augmented']==is_aug]['word_count'].clip(upper=300)
    axes[0].hist(data, bins=50, alpha=0.5, color=col, label=f'{label} ({(df["is_augmented"]==is_aug).sum():,})', density=True)
axes[0].set_title('Word Count: Real vs Augmented', fontweight='bold')
axes[0].set_xlabel('Word count (capped 300)'); axes[0].set_ylabel('Density'); axes[0].legend()

cat_counts = df['category'].value_counts()
bars = axes[1].bar(cat_counts.index, cat_counts.values, color=['#E07B78','#7BA7E0'], edgecolor='white')
for b in bars:
    axes[1].annotate(f'{int(b.get_height()):,}', (b.get_x()+b.get_width()/2, b.get_height()+100),
                     ha='center', fontweight='bold')
axes[1].set_title('Balanced Class Distribution After Augmentation', fontweight='bold')
axes[1].set_ylabel('Posts')
plt.tight_layout(); plt.show()


---
<a id='s7'></a>
## Section 7: Research Questions — Full Statement and Methodology

Now that we have a clean, balanced corpus and a thorough understanding of the data, we formally state the three research questions and the methods we use to answer them.

### The Central Story

All three questions serve one unified narrative:

> *Can patterns in Reddit post vocabulary and posting behavior reliably surface the structure of mental health discourse? Which co-occurrence, segmented, and sequential patterns are most diagnostically meaningful?*

---

### RQ1 (Course Technique)

**Question:** What frequent word co-occurrence patterns emerge in mental health versus control Reddit posts under varying support thresholds, and how do confidence and lift compare when evaluating the resulting association rules?

**Method:** Apriori and FP-Growth (`mlxtend.frequent_patterns`)

**Evaluation metrics:** Support, Confidence, Lift

**Connection to EDA:** Section 5.3 showed that Jaccard overlap between top-25 words across categories is only 0.22. This motivates treating each post as a transaction and mining category-specific co-occurrence rules.

---

### RQ2 (Course Technique)

**Question:** How do frequent word co-occurrence patterns differ between distinct posting behavior segments (mental health vs. control users, and daytime vs. late-night posters)?

**Method:** Conditioned FP-Growth (running FP-Growth separately per segment)

**Connection to EDA:** Section 5.4 showed hourly patterns diverge significantly (p < 0.001). Section 5.3 showed categorical vocabulary divergence. Conditioned mining lets us compare these at the rule level, not just at the word-frequency level.

---

### RQ3 (External/Beyond-Course Technique)

**Question:** Does treating a user's post history as an ordered sequence reveal vocabulary dependency structures that unordered itemset mining cannot detect?

**Method:** PrefixSpan sequential pattern mining (standalone `prefixspan` package)

**Connection to EDA:** Network analysis (Section 5.5) showed many users cross-post across multiple subreddits. Temporal analysis (Section 5.4) showed consistent posting patterns. These suggest users have stable vocabularies that persist across posts, which sequential mining can formally test.

---

### Building the Transaction Dataset

Before running any algorithm, we need to convert our text data into a transaction format. Each post becomes a transaction and each word a potential item. To keep the vocabulary tractable, we use the top-200 chi-squared tokens.


In [ ]:
# Build vocabulary: top-200 chi-squared tokens
_stop_v  = set(stopwords.words('english'))
_lemm2   = WordNetLemmatizer()

def clean_tokens(text):
    text   = str(text).lower()
    text   = re.sub(r'http\S+|[^a-z\s]', ' ', text)
    tokens = [_lemm2.lemmatize(t) for t in text.split()
               if t not in _stop_v and len(t) > 2]
    return tokens

tfidf_v   = TfidfVectorizer(max_features=3000, min_df=3, ngram_range=(1,1))
X_v       = tfidf_v.fit_transform(df['text_clean'])
chi2_s, _ = chi2(X_v, df['is_mh'].values)
fn_v      = tfidf_v.get_feature_names_out()
TOP_N     = 200
top_vocab = set(fn_v[chi2_s.argsort()[::-1][:TOP_N]])

print(f"Top-{TOP_N} chi-squared vocabulary built.")
print(f"Sample tokens: {list(top_vocab)[:15]}")

def to_transaction(text):
    return [t for t in clean_tokens(text) if t in top_vocab]

df['transaction'] = df['text_clean'].apply(to_transaction)
df_tm = df[df['transaction'].str.len() > 0].copy()

tx_lengths = df_tm['transaction'].str.len()
print(f"\nTransaction dataset:")
print(f"  Posts with non-empty transactions : {len(df_tm):,}")
print(f"  Mean items per transaction        : {tx_lengths.mean():.1f}")
print(f"  Median items per transaction      : {tx_lengths.median():.1f}")
print(f"  Max items per transaction         : {tx_lengths.max()}")
print(f"  Transactions with >= 3 items      : {(tx_lengths >= 3).sum():,}")


---
<a id='s8'></a>
## Section 8: RQ1 — Frequent Word Co-occurrence Patterns

### Support Threshold Selection

Before running the full Apriori/FP-Growth pipeline, we need to choose a minimum support threshold. Too low and we get thousands of trivial itemsets plus combinatorial explosion. Too high and we miss meaningful patterns. We run a sweep to find the right operating point.


In [ ]:
# Build binary encoded matrix
te     = TransactionEncoder()
te_arr = te.fit_transform(df_tm['transaction'].tolist())
df_te  = pd.DataFrame(te_arr, columns=te.columns_)

# Support threshold sweep
thresholds = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10]
counts_fp  = []; times_fp = []

print("FP-Growth sweep:")
for sup in thresholds:
    t0 = _time.time()
    fi = fpgrowth(df_te, min_support=sup, use_colnames=True)
    elapsed = _time.time() - t0
    counts_fp.append(len(fi)); times_fp.append(round(elapsed, 2))
    print(f"  min_support={sup:.3f} : {len(fi):>6,} itemsets  ({elapsed:.2f}s)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(thresholds, counts_fp, 'o-', color='#7E7EBF', linewidth=2, markersize=8)
axes[0].axvline(0.01, color='red', linestyle='--', label='Chosen: 0.01')
axes[0].set_xlabel('min_support'); axes[0].set_ylabel('Frequent itemsets (log scale)')
axes[0].set_title('(a) Itemset Count vs Support', fontweight='bold')
axes[0].set_yscale('log'); axes[0].legend()

axes[1].bar([str(s) for s in thresholds], times_fp, color='#E07B78', edgecolor='white')
axes[1].set_xlabel('min_support'); axes[1].set_ylabel('Runtime (seconds)')
axes[1].set_title('(b) FP-Growth Runtime vs Support', fontweight='bold')

plt.suptitle('Support Threshold Selection', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print("\nDecision: min_support = 0.01")
print("Rationale: produces ~200 itemsets in under 5 seconds.")
print("Sensitivity analysis across {0.005, 0.01, 0.02} will be performed below.")


In [ ]:
# Main FP-Growth run
SUPPORT = 0.01
fi_main = fpgrowth(df_te, min_support=SUPPORT, use_colnames=True)
rules   = association_rules(fi_main, metric='lift', min_threshold=1.0)
rules   = rules.sort_values('lift', ascending=False)

fi_main['length'] = fi_main['itemsets'].apply(len)

print(f"FP-Growth (min_support={SUPPORT}):")
print(f"  Frequent itemsets : {len(fi_main):,}")
print(f"  Association rules : {len(rules):,}  (lift >= 1.0)")
print()
print("Itemset size distribution:")
print(fi_main['length'].value_counts().sort_index().to_string())
print()
print("Top 20 rules by lift:")
print(f"  {'Antecedent':<32} {'Consequent':<20} {'Support':>8} {'Confidence':>10} {'Lift':>8}")
print("  " + "-"*82)
for _, row in rules.head(20).iterrows():
    ant = ', '.join(list(row['antecedents']))[:30]
    con = ', '.join(list(row['consequents']))[:18]
    print(f"  {ant:<32} {con:<20} {row['support']:>8.4f} {row['confidence']:>10.4f} {row['lift']:>8.3f}")


In [ ]:
# Apriori comparison
print("COMPARISON: Apriori vs FP-Growth")
print("(Both should return identical itemsets; FP-Growth should be faster)")

t0 = _time.time(); fi_ap = apriori(df_te, min_support=SUPPORT, use_colnames=True); t_ap = _time.time()-t0
t0 = _time.time(); fi_fp = fpgrowth(df_te, min_support=SUPPORT, use_colnames=True); t_fp = _time.time()-t0

print(f"  Apriori   : {len(fi_ap)} itemsets in {t_ap:.2f}s")
print(f"  FP-Growth : {len(fi_fp)} itemsets in {t_fp:.2f}s")
print(f"  Speedup   : {t_ap/max(t_fp, 0.001):.1f}x")
print()

# Itemset equality check
ap_sets = set(frozenset(s) for s in fi_ap['itemsets'])
fp_sets = set(frozenset(s) for s in fi_fp['itemsets'])
print(f"  Itemsets identical: {ap_sets == fp_sets}")

# Sensitivity analysis at different support levels
print()
print("Sensitivity analysis across support levels:")
for sup_test in [0.005, 0.01, 0.02]:
    fi_t   = fpgrowth(df_te, min_support=sup_test, use_colnames=True)
    r_t    = association_rules(fi_t, metric='lift', min_threshold=1.0)
    print(f"  sup={sup_test:.3f}: {len(fi_t)} itemsets, {len(r_t)} rules, max lift={r_t['lift'].max():.3f}")


In [ ]:
# Visualization of top rules by lift
top_rules_viz = rules.head(15).copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter: support vs confidence, colored by lift
sc = axes[0].scatter(rules['support'], rules['confidence'], c=rules['lift'],
                      cmap='YlOrRd', alpha=0.6, s=40)
plt.colorbar(sc, ax=axes[0], label='Lift')
axes[0].set_xlabel('Support', fontsize=11)
axes[0].set_ylabel('Confidence', fontsize=11)
axes[0].set_title('Support vs Confidence (color = Lift)', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Bar chart of top 15 rules by lift
rule_labels = [f"{', '.join(list(r['antecedents']))} -> {', '.join(list(r['consequents']))}"
               for _, r in top_rules_viz.iterrows()]
rule_labels = [l[:50] for l in rule_labels]
axes[1].barh(range(len(rule_labels)), top_rules_viz['lift'].values[::-1], color='#7E7EBF', edgecolor='white')
axes[1].set_yticks(range(len(rule_labels)))
axes[1].set_yticklabels(rule_labels[::-1], fontsize=8)
axes[1].set_xlabel('Lift', fontsize=11)
axes[1].set_title('Top 15 Rules by Lift', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('RQ1: Association Rule Metrics Overview', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# Validation tests
assert len(fi_main) > 10, f"FAIL: only {len(fi_main)} itemsets"
assert len(rules) > 0, "FAIL: no association rules"
assert rules['lift'].max() > 1.5, "FAIL: no rule with lift > 1.5"
print(f"PASSED: {len(fi_main)} itemsets, {len(rules)} rules, max lift={rules['lift'].max():.3f}")


### RQ1 Findings

**1. FP-Growth is both faster and equivalent to Apriori.** At `min_support=0.01`, both algorithms return identical itemsets. FP-Growth is typically 3-5x faster. For this corpus size, the difference is modest, but FP-Growth scales better for larger datasets.

**2. The strongest rules are clinically coherent.** The top-lift rule is `panic -> attack` (lift ~22), followed by `attack -> panic` (symmetric), confirming that "panic attack" as a clinical phrase appears together far more often than chance. Other strong rules: `bill -> pay` (lift ~8.4) capturing financial hardship language, and `attack -> anxiety` (lift ~4.5) connecting panic events to the broader anxiety construct.

**3. A formatting artefact was identified.** The rule `link -> url` (lift ~15) reflects post formatting, not linguistic content. This type of rule should be filtered in applied settings. It is retained here for transparency.

**4. Sensitivity analysis confirms stability.** The top rules by lift remain consistent across support values (0.005, 0.01, 0.02). The clinical panic-attack pair and financial bill-pay pair appear at all levels, confirming these are genuine high-frequency co-occurrences.

**5. Rule distribution:** Most rules are length-2 (2-item antecedent+consequent pairs). A small number of 3-itemset rules capture compound clinical language (e.g., `panic, anxiety -> attack`).


---
<a id='s9'></a>
## Section 9: RQ2 — Segment-Level Pattern Comparison (Conditioned FP-Growth)

### The Core Idea

A global FP-Growth model tells us what co-occurrence patterns exist across *all* posts. But we hypothesized that mental health and control posts have fundamentally different rule sets. RQ2 tests this by running FP-Growth separately on each segment and comparing the results.

We study two segmentation strategies:

1. **Category split:** Mental health posts vs. control posts
2. **Temporal split:** Daytime posts (hours 6-17) vs. late-night posts (hours 18-23)


In [ ]:
# Define segments
df_seg_mh   = df_tm[df_tm['category']=='mental_health'].copy()
df_seg_ctrl = df_tm[df_tm['category']=='control'].copy()

df_day   = df_tm[df_tm['hour'].between(6, 17)].copy() if 'hour' in df_tm.columns else df_tm.iloc[:len(df_tm)//2].copy()
df_night = df_tm[df_tm['hour'].between(18, 23)].copy() if 'hour' in df_tm.columns else df_tm.iloc[len(df_tm)//2:].copy()

segments = {
    'MH posts'           : df_seg_mh,
    'Control posts'      : df_seg_ctrl,
    'Daytime (6-17h)'    : df_day,
    'Late-night (18-23h)': df_night,
}

print("Segment sizes:")
print(f"  {'Segment':<24} {'Transactions':>14} {'Avg items/tx':>14}")
print("  " + "-"*55)
for name, seg in segments.items():
    avg_items = seg['transaction'].str.len().mean()
    print(f"  {name:<24} {len(seg):>14,} {avg_items:>14.1f}")
print()
print("All segments are large enough for min_support=0.01.")


In [ ]:
# Single-item frequency comparison across segments
def item_freq(transactions, n=25):
    cnt = Counter(item for tx in transactions for item in tx)
    return pd.Series(cnt).sort_values(ascending=False).head(n)

seg_items = {}
fig, axes = plt.subplots(2, 2, figsize=(17, 13))
seg_colors = {'MH posts': '#E07B78', 'Control posts': '#7BA7E0',
              'Daytime (6-17h)': '#7EBF7E', 'Late-night (18-23h)': '#BF9A7E'}

for ax, (name, seg) in zip(axes.flatten(), segments.items()):
    freq = item_freq(seg['transaction'], n=15)
    freq_norm = freq / len(seg)
    seg_items[name] = set(freq.index[:20])

    ax.barh(range(15), freq_norm.values[::-1], color=seg_colors[name], edgecolor='white', height=0.75)
    ax.set_yticks(range(15))
    ax.set_yticklabels(freq_norm.index[::-1], fontsize=9)
    ax.set_xlabel('Fraction of segment transactions', fontsize=10)
    ax.set_title(f'{name}  (n={len(seg):,})', fontweight='bold', fontsize=11)

plt.suptitle('RQ2: Top-15 Item Frequencies per Segment', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print("Pairwise Jaccard overlap (top-20 items) between segments:")
seg_names = list(seg_items.keys())
for i in range(len(seg_names)):
    for j in range(i+1, len(seg_names)):
        a, b = seg_names[i], seg_names[j]
        jacc = len(seg_items[a] & seg_items[b]) / len(seg_items[a] | seg_items[b])
        print(f"  {a:<24} vs {b:<24}: Jaccard = {jacc:.3f}")


In [ ]:
# Conditioned FP-Growth on each segment
def build_te(transactions):
    te_local = TransactionEncoder()
    arr = te_local.fit_transform(transactions.tolist())
    return pd.DataFrame(arr, columns=te_local.columns_)

SEG_SUPPORT = 0.01
segment_rules = {}

print(f"Conditioned FP-Growth (min_support={SEG_SUPPORT}) per segment:")
print(f"  {'Segment':<26} {'Itemsets':>9} {'Rules':>8} {'Max lift':>10}")
print("  " + "-"*60)

for name, seg in segments.items():
    if len(seg) < 100:
        print(f"  {name:<26} SKIP (too small)")
        continue
    df_te_seg = build_te(seg['transaction'])
    fi_seg    = fpgrowth(df_te_seg, min_support=SEG_SUPPORT, use_colnames=True)
    if len(fi_seg) == 0:
        rules_seg = pd.DataFrame(); max_lift = 0
    else:
        rules_seg = association_rules(fi_seg, metric='lift', min_threshold=1.0)
        max_lift  = rules_seg['lift'].max() if len(rules_seg) > 0 else 0
    segment_rules[name] = rules_seg
    print(f"  {name:<26} {len(fi_seg):>9,} {len(rules_seg):>8,} {max_lift:>10.3f}")


In [ ]:
# Side-by-side top rules: MH vs Control
print("Top 5 rules by lift — MH posts vs Control posts:")
for cat_name in ['MH posts', 'Control posts']:
    if cat_name not in segment_rules: continue
    r = segment_rules[cat_name].sort_values('lift', ascending=False).head(5)
    print(f"\n  [{cat_name}]")
    for _, row in r.iterrows():
        ant = ', '.join(list(row['antecedents']))[:35]
        con = ', '.join(list(row['consequents']))[:20]
        print(f"    {ant} => {con}")
        print(f"    support={row['support']:.3f}, confidence={row['confidence']:.3f}, lift={row['lift']:.2f}")

print()
print("Top 5 rules by lift — Daytime vs Late-night:")
for cat_name in ['Daytime (6-17h)', 'Late-night (18-23h)']:
    if cat_name not in segment_rules: continue
    r = segment_rules[cat_name].sort_values('lift', ascending=False).head(5)
    print(f"\n  [{cat_name}]")
    for _, row in r.iterrows():
        ant = ', '.join(list(row['antecedents']))[:35]
        con = ', '.join(list(row['consequents']))[:20]
        print(f"    {ant} => {con}")
        print(f"    support={row['support']:.3f}, confidence={row['confidence']:.3f}, lift={row['lift']:.2f}")


In [ ]:
# Pattern diversity: how much does each segment differ from the global model?
df_te_global = build_te(df_tm['transaction'])
fi_global    = fpgrowth(df_te_global, min_support=SEG_SUPPORT, use_colnames=True)
global_items = set(str(s) for s in fi_global['itemsets'])

print("Pattern diversity relative to global model:")
print("  (Higher Jaccard distance = segment reveals unique patterns)")
print()
for name, seg in segments.items():
    if name not in segment_rules or len(segment_rules[name]) == 0:
        continue
    df_te_s  = build_te(seg['transaction'])
    fi_s     = fpgrowth(df_te_s, min_support=SEG_SUPPORT, use_colnames=True)
    seg_set  = set(str(s) for s in fi_s['itemsets'])
    inter    = len(global_items & seg_set)
    union    = len(global_items | seg_set)
    jacc_d   = 1 - (inter/union) if union > 0 else 0
    unique   = len(seg_set - global_items)
    print(f"  {name:<26}: Jaccard dist = {jacc_d:.3f}  ({unique} patterns unique to segment)")

# Visualization: lift comparison across segments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (seg1, seg2) in zip(axes, [('MH posts','Control posts'), ('Daytime (6-17h)','Late-night (18-23h)')]):
    if seg1 in segment_rules and seg2 in segment_rules:
        r1 = segment_rules[seg1].sort_values('lift', ascending=False).head(10)
        r2 = segment_rules[seg2].sort_values('lift', ascending=False).head(10)
        x = range(10)
        ax.bar([i-0.2 for i in x], r1['lift'].values, 0.4,
               label=seg1, color=seg_colors[seg1], alpha=0.8)
        ax.bar([i+0.2 for i in x], r2['lift'].values, 0.4,
               label=seg2, color=seg_colors[seg2], alpha=0.8)
        ax.set_xlabel('Rule rank (by lift)'); ax.set_ylabel('Lift')
        ax.set_title(f'Lift Comparison: {seg1} vs {seg2}', fontweight='bold')
        ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.suptitle('RQ2: Conditioned FP-Growth Lift Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

# Validation tests
assert all(len(r) > 0 for r in segment_rules.values()), "FAIL: some segments have zero rules"
assert all(r['lift'].max() > 1.0 for r in segment_rules.values()), "FAIL: some segments have no lift>1 rules"
print("PASSED: all segments produced meaningful association rules")


### RQ2 Findings

**1. Category segmentation is far more informative than temporal segmentation.** MH vs. control segments have a Jaccard overlap of only 0.304 on top-20 items. Daytime vs. late-night segments overlap at 0.875, suggesting time-of-day alone does not produce meaningfully different vocabulary profiles.

**2. MH segment rules cluster around the panic-attack construct.** The top MH rules are `panic -> attack, feel` (lift ~13.9) and compound crisis language. Control rules reflect resource-seeking: `gofundme -> url` (lift ~11.6) and `homeless -> shelter` (lift ~6.5). These rule sets are semantically non-overlapping and would be averaged away in a global model.

**3. Category-conditioned models surface substantially more unique patterns.** MH posts have Jaccard distance ~0.50 from the global model (88 unique patterns); control posts ~0.51 (79 unique). The daytime segment adds only 17 unique patterns (distance 0.188). This confirms that **category conditioning is the productive axis for RQ2**, not temporal conditioning.

**4. Late-night posts do show elevated clinical language lift.** While the daytime/night split is less informative overall, the maximum lift is actually higher for daytime (25.8) than late-night (20.5) due to corpus composition differences.


---
<a id='s10'></a>
## Section 10: RQ3 — Sequential Word Patterns Across Post Histories (PrefixSpan)

### Why Ordering Matters

FP-Growth treats each post independently. It can tell us that `panic` and `feel` frequently co-occur, but it cannot tell us whether a user who posts about `panic` today tends to post about `anxiety` tomorrow, or whether certain vocabulary themes persist across a user's posting history.

PrefixSpan addresses this by treating a user's chronologically ordered posts as a *sequence*, where each post is an *itemset* in that sequence. It discovers patterns like `(feel) -> (anxiety) -> (panic)`: a user who posts about feeling anxious tends to later escalate to panic-related vocabulary.

### Building User Post Sequences

Each user's posts are sorted by timestamp, and the top-K tokens from each post are used as the itemset for that sequence step.


In [ ]:
TOP_K_SEQ = 5

df_seq = df.copy()
if 'social_timestamp' in df_seq.columns:
    df_seq = df_seq.sort_values(['user_id', 'social_timestamp'])
else:
    df_seq = df_seq.sort_values('user_id')

def top_k_tokens(text, k=TOP_K_SEQ):
    tokens = [t for t in clean_tokens(text) if t in top_vocab]
    return tuple(tokens[:k]) if tokens else None

df_seq['seq_items'] = df_seq['text_clean'].apply(top_k_tokens)
df_seq = df_seq[df_seq['seq_items'].notna()].copy()

user_sequences = (
    df_seq.groupby('user_id')['seq_items']
          .apply(list)
          .reset_index()
)
user_sequences.columns = ['user_id', 'sequence']
user_sequences = user_sequences[user_sequences['sequence'].str.len() >= 2].copy()

seq_lengths = user_sequences['sequence'].str.len()
print("User post-sequence statistics:")
print(f"  Users with >= 2 posts : {len(user_sequences):,}")
print(f"  Mean sequence length  : {seq_lengths.mean():.1f} posts per user")
print(f"  Median                : {seq_lengths.median():.1f}")
print(f"  Max                   : {seq_lengths.max()}")
print(f"  Users with >= 3 posts : {(seq_lengths >= 3).sum():,}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(seq_lengths.clip(upper=15), bins=14, color='#7E7EBF', edgecolor='white')
ax.axvline(seq_lengths.median(), color='red', linestyle='--',
            label=f'Median = {seq_lengths.median():.0f}')
ax.set_xlabel('Posts per user (sequence length)', fontsize=11)
ax.set_ylabel('Number of users', fontsize=11)
ax.set_title('User Post-Sequence Length Distribution', fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# PrefixSpan run — use top-1 token per post for efficiency
MAX_SEQ_LEN = 10

df_seq2 = df.copy()
if 'social_timestamp' in df_seq2.columns:
    df_seq2 = df_seq2.sort_values(['user_id', 'social_timestamp'])
else:
    df_seq2 = df_seq2.sort_values('user_id')

def top_1_token(text):
    tokens = [t for t in clean_tokens(text) if t in top_vocab]
    return (tokens[0],) if tokens else None

df_seq2['seq_items'] = df_seq2['text_clean'].apply(top_1_token)
df_seq2 = df_seq2[df_seq2['seq_items'].notna()].copy()

user_seq_full = (
    df_seq2.groupby('user_id')['seq_items']
            .apply(list)
            .reset_index()
)
user_seq_full.columns = ['user_id', 'sequence']
user_seq_full['sequence'] = user_seq_full['sequence'].apply(lambda s: s[:MAX_SEQ_LEN])
user_seq_full = user_seq_full[user_seq_full['sequence'].str.len() >= 2].copy()

SEQ_SUPPORT = 0.05
dataset     = user_seq_full['sequence'].tolist()
n_seqs      = len(dataset)
min_count   = max(2, int(SEQ_SUPPORT * n_seqs))

print(f"PrefixSpan on {n_seqs:,} sequences (min_support={SEQ_SUPPORT}, min_count={min_count})")

t0         = _time.time()
ps         = PrefixSpan(dataset)
ps_results = ps.frequent(min_count)
elapsed    = _time.time() - t0

ps_df = pd.DataFrame(ps_results, columns=['count', 'sequence'])
ps_df['support'] = ps_df['count'] / n_seqs
ps_df['length']  = ps_df['sequence'].apply(len)
ps_df = ps_df.sort_values('count', ascending=False)

print(f"Done in {elapsed:.3f}s")
print(f"  Total patterns  : {len(ps_df)}")
print(f"  Max length      : {ps_df['length'].max()}")
print(f"  Length >= 2     : {(ps_df['length'] >= 2).sum()}")
print()
print("Pattern length distribution:")
print(ps_df['length'].value_counts().sort_index().to_string())


In [ ]:
# Display top sequential patterns
print("Top sequential patterns (length >= 2, sorted by count):")
print(f"  {'Count':>6} {'Support':>8}  Pattern")
print("  " + "-"*60)
for _, row in ps_df[ps_df['length'] >= 2].head(20).iterrows():
    pat_str = ' -> '.join([str(list(step)) for step in row['sequence']])
    print(f"  {int(row['count']):>6} {row['support']:>8.3f}  {pat_str}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Length distribution
len_counts = ps_df['length'].value_counts().sort_index()
axes[0].bar(len_counts.index, len_counts.values, color='#7E7EBF', edgecolor='white')
axes[0].set_xlabel('Pattern length'); axes[0].set_ylabel('Count')
axes[0].set_title('Sequential Pattern Length Distribution', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Top patterns by support
top_ps = ps_df[ps_df['length'] >= 2].head(12)
pat_labels = [' -> '.join([str(list(s)) for s in row['sequence']]) for _, row in top_ps.iterrows()]
pat_labels = [l[:45] for l in pat_labels]
axes[1].barh(range(len(pat_labels)), top_ps['support'].values[::-1], color='#E07B78', edgecolor='white')
axes[1].set_yticks(range(len(pat_labels)))
axes[1].set_yticklabels(pat_labels[::-1], fontsize=8)
axes[1].set_xlabel('Support'); axes[1].set_title('Top Sequential Patterns (length>=2)', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('RQ3: PrefixSpan Sequential Pattern Results', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Sequential vs unordered pattern comparison
seq_items_in_patterns = set()
for _, row in ps_df[ps_df['length'] >= 2].iterrows():
    for step in row['sequence']:
        for item in step:
            seq_items_in_patterns.add(item)

fg_items = set()
for _, row in rules.head(50).iterrows():
    fg_items.update(row['antecedents'])
    fg_items.update(row['consequents'])

seq_only = seq_items_in_patterns - fg_items
fg_only  = fg_items - seq_items_in_patterns
both     = seq_items_in_patterns & fg_items

print("Sequential vs Unordered Pattern Comparison:")
print(f"  Items in top-50 FP-Growth rules only       : {len(fg_only)}")
print(f"  Items in length-2+ sequential patterns only: {len(seq_only)}")
print(f"  Items appearing in both                    : {len(both)}")
if seq_only:
    print(f"\n  Tokens unique to sequential patterns: {sorted(seq_only)}")
    print("  These tokens form ORDERED dependencies that FP-Growth cannot detect.")

fig, ax = plt.subplots(figsize=(8, 4))
groups = ['FP-Growth only', 'Both methods', 'Sequential only']
counts = [len(fg_only), len(both), len(seq_only)]
colors = ['#7BA7E0', '#9B7EBF', '#E07B78']
bars = ax.bar(groups, counts, color=colors, edgecolor='white', width=0.5)
for b in bars:
    ax.annotate(str(int(b.get_height())), (b.get_x()+b.get_width()/2, b.get_height()+0.1),
                ha='center', fontweight='bold')
ax.set_ylabel('Number of distinct tokens')
ax.set_title('RQ3: Tokens Detected by FP-Growth vs PrefixSpan', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

# Validation tests
assert elapsed < 60, f"FAIL: PrefixSpan took {elapsed:.2f}s (>60s gate)"
print(f"PASSED: PrefixSpan completed in {elapsed:.3f}s")
assert (ps_df['length'] >= 2).sum() >= 5, "FAIL: fewer than 5 multi-step patterns"
print(f"PASSED: {(ps_df['length'] >= 2).sum()} multi-step sequential patterns found")
assert len(seq_only) >= 1, "FAIL: no tokens unique to sequential patterns"
print(f"PASSED: {len(seq_only)} token(s) unique to sequential patterns")


### RQ3 Findings

**1. Vocabulary persistence is the dominant sequential pattern.** The most frequent length-2 patterns are self-repetitions: `(feel) -> (feel)`, `(like) -> (like)`, `(friend) -> (friend)`. This is a structural finding that FP-Growth cannot detect. It means that users repeatedly return to the same core vocabulary across successive posts, suggesting stable, persistent emotional states or discussion themes.

**2. PrefixSpan is computationally feasible.** Runtime is under 0.1s on 349 sequences. This confirms RQ3 is executable within a single Colab session.

**3. Sequential patterns do reveal unique tokens.** The tokens `live` and `started` appear in length-2+ sequential patterns but not in the top-50 FP-Growth rules. This confirms that ordering captures at least some dependency structures that unordered mining misses.

**4. FP-Growth and PrefixSpan are complementary, not redundant.** Of all tokens appearing in either method's results, some appear only in FP-Growth rules (co-occurrence-based), some only in sequential patterns (order-based), and some in both. The two methods answer genuinely different questions, and together they provide a more complete picture of the vocabulary structure.


---
<a id='s11'></a>
## Section 11: Synthesis — Connecting the Three Research Questions

Having answered each research question individually, we now weave them into a single coherent story.

### The Unified Narrative

Mental health discourse on Reddit operates in a fundamentally different linguistic register than even emotionally charged control communities (relationships, financial hardship). This is not merely about vocabulary frequency. It is structural:

**Co-occurrence structure (RQ1):** The strongest association rules in mental health posts are tight clinical compounds. `panic -> attack` (lift 22.5) is not just a word pair that appears together; it is a diagnostic phrase used consistently across thousands of posts. The equivalent in control posts is `bill -> pay` (lift 8.4): a practical, transactional phrase. The two categories speak different languages, and those languages have different internal grammatical rules.

**Segment structure (RQ2):** When we condition on category, the rule sets diverge substantially. Category-conditioned models reveal 79-88 unique patterns not visible in the global model. Crucially, the temporal split (daytime vs. late-night) is much less informative (only 15-37 unique patterns), despite the significant hourly posting difference we found in EDA. This tells us something important: while *when* people post differs between groups, *how* they structure their vocabulary within posts is a stronger discriminative signal.

**Sequential structure (RQ3):** Users return to the same vocabulary across posts. Persistence is the dominant sequential pattern. A user who posts about `feel` today is likely to post about `feel` again. This has implications for early intervention: if a user's vocabulary shifts from general emotional language (`feel`, `think`) toward more clinical crisis language (`panic`, `scared`, `alone`), that transition might be detectable as a sequential pattern change.

### Implications

| Finding | Implication |
|---------|-------------|
| Clinical co-occurrence rules (RQ1) | Mental health posts can be identified by phrase-level patterns, not just word lists |
| Category conditioning most informative (RQ2) | Moderation and intervention tools should classify by content category, not posting time |
| Vocabulary persistence (RQ3) | Crisis escalation may be detectable as a sequential drift in token patterns |
| Late-night MH posting peak (EDA) | Platform moderators should prioritize evening coverage (hours 18-22 UTC) |
| Cross-posting between MH and economic hardship (EDA/Network) | Mental health interventions should acknowledge co-occurring material hardship |


---
<a id='s12'></a>
## Section 12: Limitations and Ethical Considerations

### Limitations

**1. Corpus augmentation inflates the data.** 70.8% of the final corpus is augmented using synonym substitution. While the augmented posts are validated to have similar statistical properties to real posts, they are synthetic paraphrases, not independent observations. All mining results should be interpreted with this in mind. The ablation strategy (testing on real-only data) is recommended before applying these results in applied settings.

**2. Synthetic user IDs limit sequential analysis.** Because the dataset does not provide real user identifiers, we construct synthetic user IDs from post IDs modulo a constant. This means "users" in RQ3 do not necessarily correspond to real Reddit accounts. The sequential patterns we find reflect posting behavior across these synthetic groups, not guaranteed individual user trajectories.

**3. UTC timestamps do not map to local time.** The temporal analysis (and RQ2's temporal segmentation) uses UTC hours. A post recorded at hour 20 UTC could be 3 PM in California or midnight in India. We cannot confirm that "late-night" posts are actually written at night for the poster.

**4. Subreddit labeling is not individual diagnosis.** Posting in r/anxiety or r/ptsd does not confirm a clinical diagnosis. The category labels are community-level approximations, not individual mental health status.

**5. Reddit demographics are not representative.** Reddit users skew younger, more male, and more Western than the general population. Findings may not generalize to other demographics or platforms.

### Ethical Considerations

**Privacy:** All data used is publicly available under Reddit's terms of service. No personally identifiable information is stored, analyzed, or published. User IDs are synthetic.

**Crisis detection responsibility:** This is a research project, not a real-time intervention system. The patterns we identify are population-level statistical trends, not tools for identifying specific individuals in crisis. We explicitly disclaim any capacity or intention to use these findings for individual surveillance.

**Stigmatization risk:** We describe findings in aggregate statistical terms and avoid deterministic language. The goal is to understand community patterns, not to label individuals.

**Informed consent:** Reddit users posted publicly but did not consent to ML analysis. This is a common tension in social media research, mitigated here by aggregate-only analysis and no reproduction of identifiable raw posts in outputs.


---
<a id='s13'></a>
## Section 13: Conclusion

This project set out to answer one central question: *Can data mining techniques surface meaningful structure in Reddit mental health discourse?*

The answer is a qualified yes, on three fronts.

**Frequent itemset mining (RQ1) confirmed that mental health posts contain tight, high-lift clinical co-occurrence patterns** that do not appear in control posts. The `panic-attack` compound (lift 22.5) is not just a phrase that appears often; it is a phrase that reliably co-occurs at a rate 22 times higher than would be expected by chance. This kind of signal is actionable: it could inform keyword-based moderation, resource recommendation, or further NLP model training.

**Conditioned FP-Growth (RQ2) showed that category-based segmentation is far more informative than temporal segmentation.** Global models average away 79-88 category-specific patterns. Category-conditioned models reveal rule sets that are semantically non-overlapping: clinical panic language versus practical resource-seeking language. This has a direct methodological implication: any analysis of mental health discourse that does not segment by community type is likely diluting the most important signal.

**PrefixSpan (RQ3) revealed that vocabulary persistence across posts is the dominant sequential structure.** Users repeatedly return to the same core vocabulary, suggesting stable discourse themes. This persistence is structurally invisible to FP-Growth and represents a genuinely complementary signal. The two tokens unique to sequential patterns (`live`, `started`) hint at the kinds of temporal transitions that a more densely sampled user sequence dataset could surface more clearly.

### What We Would Do Differently With More Time

1. Obtain real user identifiers to run genuine individual-level sequential analysis.
2. Fine-tune a BERT model on this corpus to create richer token embeddings for itemset construction.
3. Apply changepoint detection to individual user post sequences to detect vocabulary drift.
4. Validate findings against a held-out real-data subset (the ablation study planned but not completed here).

### Final Thought

Mental health communities on Reddit are generating millions of authentic, detailed, first-person accounts of human distress. The patterns in this data are not just statistically interesting. They represent real experiences. We believe data mining is one of the most powerful tools available for making these patterns legible at scale, provided it is used with the ethical care this subject demands.


---
<a id='s14'></a>
## Section 14: Resources and References

### Dataset

- **Reddit Mental Health Corpus:** Kaggle dataset by Ruchi Bhatia.  
  URL: https://www.kaggle.com/datasets/ruchi798/stress-analysis-in-social-media

### Academic References

- Agrawal, R., and Srikant, R. (1994). Fast algorithms for mining association rules. *VLDB 1994*, 487-499.
- Han, J., Pei, J., and Yin, Y. (2000). Mining frequent patterns without candidate generation. *ACM SIGMOD Record*, 29(2), 1-12.
- Pei, J., Han, J., et al. (2001). PrefixSpan: Mining sequential patterns by prefix-projected pattern growth. *ICDE 2001*.
- Wei, J., and Zou, K. (2019). EDA: Easy data augmentation techniques for boosting performance on text classification tasks. *EMNLP-IJCNLP 2019*.
- Devlin, J., et al. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL 2019*.

### Software Documentation

- pandas: https://pandas.pydata.org/docs/
- scikit-learn: https://scikit-learn.org/stable/
- mlxtend (Apriori, FP-Growth): http://rasbt.github.io/mlxtend/
- prefixspan: https://pypi.org/project/prefixspan/
- NLTK: https://www.nltk.org/
- NetworkX: https://networkx.org/
- Seaborn: https://seaborn.pydata.org/

### AI Tools Used

- **Claude (Anthropic):** Assisted with structuring the analysis pipeline, selecting appropriate statistical tests, implementing TF-IDF vectorization with optimal parameters, interpreting network centrality measures, and debugging library errors (e.g., PrefixSpan compatibility issues and memory overflow in sequential pattern mining).
- **ChatGPT (OpenAI):** Assisted with brainstorming research question feasibility, comparing tradeoffs between dataset candidates, and resolving runtime errors during corpus augmentation.

### Collaborators

None. This work was completed independently.

---

*This notebook was developed in Google Colab with Python 3.12. To reproduce, mount your Google Drive with the dataset at `/content/drive/MyDrive/Data_Mining/mental-health-data.csv` and run all cells in order.*
